# Topic 12: SARSA (on-policy TD control)

SARSA keeps everything from Monte Carlo Control (Topic 11), the Q-table, epsilon-greedy
selection, the generalized-policy-iteration shell, and changes only the update: instead of
waiting for the whole-episode return, it moves Q(s,a) toward `r + gamma * Q(s', a')`, where
`a'` is the action the policy will actually take next. Using the real next action makes it
**on-policy**. The name is the quintuple it consumes: State, Action, Reward, State, Action.

## 1. The maze environment (same 5x7 GridWorld as MC Control)

In [1]:
import numpy as np, random

class GridWorld:
    def __init__(self): self.x, self.y = 2, 0
    def step(self, a):
        if a == 0: self.move_left()
        elif a == 1: self.move_up()
        elif a == 2: self.move_right()
        elif a == 3: self.move_down()
        return (self.x, self.y), -1, self.is_done()
    def move_left(self):
        if self.y == 0: pass
        elif self.y == 3 and self.x in [0,1,2]: pass
        elif self.y == 5 and self.x in [2,3,4]: pass
        else: self.y -= 1
    def move_right(self):
        if self.y == 1 and self.x in [0,1,2]: pass
        elif self.y == 3 and self.x in [2,3,4]: pass
        elif self.y == 6: pass
        else: self.y += 1
    def move_up(self):
        if self.x == 0: pass
        elif self.x == 3 and self.y == 2: pass
        else: self.x -= 1
    def move_down(self):
        if self.x == 4: pass
        elif self.x == 1 and self.y == 4: pass
        else: self.x += 1
    def is_done(self): return self.x == 4 and self.y == 6
    def reset(self): self.x, self.y = 0, 0; return (self.x, self.y)

## 2. The SARSA agent

Only `update` differs from the MC-control agent: it bootstraps off `a_prime`, the action the policy will take next.

In [2]:
class SarsaAgent:
    def __init__(self):
        self.q_table = np.zeros((5, 7, 4))   # Q(s, a): 5x7 states, 4 actions
        self.eps = 0.9
        self.alpha = 0.1
        self.gamma = 0.9

    def select_action(self, s):              # eps-greedy
        x, y = s
        if random.random() < self.eps:
            return random.randint(0, 3)      # explore
        return int(np.argmax(self.q_table[x, y, :]))   # exploit

    def update(self, s, a, r, s_prime, a_prime, done):
        # TD control update, using the action a_prime the policy WILL take.
        x, y = s
        x2, y2 = s_prime
        target = r + (0.0 if done else self.gamma * self.q_table[x2, y2, a_prime])
        self.q_table[x, y, a] += self.alpha * (target - self.q_table[x, y, a])

    def anneal_eps(self):
        self.eps = max(self.eps - 0.03, 0.1)

## 3. The SARSA loop

Choose the first action *before* stepping, then reuse each next action `a'` as the following step's `a`, that reuse is what keeps SARSA on-policy.

In [3]:
random.seed(0); np.random.seed(0)
env = GridWorld()
agent = SarsaAgent()

for n_epi in range(3000):
    s = env.reset()
    a = agent.select_action(s)               # choose the FIRST action up front
    done = False
    while not done:
        s_prime, r, done = env.step(a)        # take a
        a_prime = agent.select_action(s_prime)  # choose the NEXT action
        agent.update(s, a, r, s_prime, a_prime, done)
        s, a = s_prime, a_prime               # reuse it: on-policy
    agent.anneal_eps()

policy = np.argmax(agent.q_table, axis=2)     # greedy action per cell
print("greedy action per cell (0=L, 1=U, 2=R, 3=D):")
print(policy)

greedy action per cell (0=L, 1=U, 2=R, 3=D):
[[3 3 0 2 2 2 3]
 [2 3 0 2 2 3 3]
 [2 3 0 1 0 3 3]
 [2 2 2 1 0 3 3]
 [0 1 2 1 0 2 0]]


## 4. Read the policy as arrows and walk it

Walls are gray; we follow the greedy action from Start and check it reaches the Goal.

In [4]:
arrows = {0: '<', 1: '^', 2: '>', 3: 'v'}
walls = {(r, 2) for r in [0,1,2]} | {(r, 4) for r in [2,3,4]}
goal = (4, 6)
for r in range(5):
    row = []
    for c in range(7):
        if (r, c) in walls: row.append('##')
        elif (r, c) == goal: row.append(' G')
        else: row.append(' ' + arrows[int(policy[r, c])])
    print(''.join(row))

# follow the greedy policy from Start
s = (0, 0); path = [s]
for _ in range(60):
    if s == goal: break
    a = int(np.argmax(agent.q_table[s[0], s[1], :]))
    env.x, env.y = s
    s, _, _ = env.step(a)
    path.append(s)
print("\ngreedy path length:", len(path) - 1, "| reached goal:", s == goal)

 v v## > > > v
 > v## > > v v
 > v## ^## v v
 > > > ^## v v
 < ^ > ^## > G

greedy path length: 14 | reached goal: True


## Try it

- Turn off exploration decay (keep `eps` fixed) and watch the policy stay noisier: SARSA learns the value of whatever policy it follows.
- Change `alpha`: larger learns faster but noisier, the same step-size trade-off as TD prediction.
- Next topic: replace `self.q_table[x2, y2, a_prime]` with `np.max(self.q_table[x2, y2, :])`, and SARSA becomes **Q-Learning** (off-policy).